# FairyZero — khảo sát trần hiệu năng thật sự của T4

Phiên đo trước đã chứng minh **gom batch và chạy song song đều không phải nút thắt**.
Engine đạt ~2.200 NN eval/giây ≈ 2,2 TFLOP/s ≈ **27% đỉnh fp32 của T4**.

Sổ tay này trả lời câu còn lại: **27% đó là giới hạn của phần cứng, hay của ONNX Runtime?**

| Phần | Đo gì | Bắt buộc? |
|---|---|---|
| **A** | Suy luận **thuần tuý** mạng của bạn: CUDA EP vs TensorRT, nhiều cỡ batch | **Có — quan trọng nhất** |
| **B** | lc0 trên cùng T4: `cuda` (kernel viết tay) vs `onnx-cuda` | Tuỳ chọn |
| **C** | Quét `ev/play` trên engine bằng `--search-opt` | Có |

Phần A **không cần engine** — chỉ Python + file `.onnx`. Nó tách bạch
"tốc độ suy luận" khỏi "mọi thứ khác trong engine", điều chưa bao giờ được làm.


## 0. Chuẩn bị


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!pip -q install onnx onnxruntime-gpu 2>&1 | tail -2
%cd /content
!wget -q -nc https://github.com/phuc11731510/chess_variant_engine/releases/download/v1.0.0/12bx144fx8s_12.onnx
!ls -la 12bx144fx8s_12.onnx


---
# PHẦN A — Suy luận thuần tuý  ⬅ QUAN TRỌNG NHẤT

Không MCTS, không cây, không CPU nào khác. Chỉ nạp mạng và bắn tensor ngẫu nhiên
vào. Cho biết **trần tuyệt đối** mà mạng này đạt được trên T4.


## A1. Đếm FLOP từ chính đồ thị ONNX

Không đoán kiến trúc — đọc thẳng kích thước kernel từ file.


In [ ]:
import time, numpy as np, onnx, onnxruntime as ort

MODEL = "/content/12bx144fx8s_12.onnx"
BOARD = 10

def onnx_flops(path, board):
    """FLOP moi vi tri, doc thang tu do thi ONNX (khong doan kien truc)."""
    m = onnx.load(path)
    init = {i.name: i for i in m.graph.initializer}
    total = 0
    for n in m.graph.node:
        if not (len(n.input) > 1 and n.input[1] in init):
            continue
        w = list(init[n.input[1]].dims)
        if n.op_type == "Conv" and len(w) == 4:
            # [Cout, Cin, kh, kw]; mang nay giu nguyen kich thuoc khong gian
            total += 2 * w[0] * w[1] * w[2] * w[3] * board * board
        elif n.op_type in ("Gemm", "MatMul") and len(w) == 2:
            total += 2 * w[0] * w[1]
    return total

GF = onnx_flops(MODEL, BOARD) / 1e9
print("Mang:", MODEL)
print("FLOP moi vi tri = %.3f GFLOP" % GF)
print("ORT", ort.__version__)
print("providers co san:", ort.get_available_providers())


## A2. CUDA EP — đúng ngăn xếp engine đang dùng


In [ ]:
def bench(provider, batches, iters=30, warmup=8, popts=None):
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    plist = [(provider, popts)] if popts else [provider]
    try:
        sess = ort.InferenceSession(MODEL, so, providers=plist)
    except Exception as e:
        print("  %-28s KHONG KHA DUNG: %s" % (provider, str(e)[:90]))
        return {}
    if provider not in sess.get_providers():
        print("  %-28s KHONG kich hoat duoc (dang chay %s)" % (provider, sess.get_providers()))
        return {}
    iname = sess.get_inputs()[0].name
    res = {}
    for b in batches:
        x = np.random.rand(b, 226, BOARD, BOARD).astype(np.float32)
        try:
            for _ in range(warmup):
                sess.run(None, {iname: x})
            t0 = time.perf_counter()
            for _ in range(iters):
                sess.run(None, {iname: x})
            dt = (time.perf_counter() - t0) / iters
        except Exception as e:
            print("  %-28s batch=%-4d LOI: %s" % (provider, b, str(e)[:70]))
            continue
        res[b] = b / dt
        print("  %-28s batch=%-4d %9.1f pos/s   %6.2f TFLOP/s" % (provider, b, b/dt, b/dt*GF))
    return res

BATCHES = [1, 8, 16, 32, 64, 128, 256]
print("=== CUDAExecutionProvider (ngan xep giong engine) ===")
cuda = bench("CUDAExecutionProvider", BATCHES)


## A3. TensorRT EP (fp32 — KHÔNG đổi độ chính xác)

⚠ TensorRT phải **biên dịch kế hoạch** riêng cho từng cỡ batch. Lần đầu mỗi cỡ
mất 1-5 phút. Cell này có thể chạy 10-20 phút — bình thường, cứ để nó chạy.


In [ ]:
# TensorRT phai BIEN DICH ke hoach rieng cho tung co batch -> lan dau rat lau
# (1-5 phut moi co). Bat cache de lan sau nhanh. Dung it co batch hon.
import os
os.makedirs("/content/trt_cache", exist_ok=True)
TRT_OPTS = {
    "trt_engine_cache_enable": True,
    "trt_engine_cache_path": "/content/trt_cache",
    "trt_fp16_enable": False,          # GIU fp32 -- khong doi do chinh xac
}
print("=== TensorrtExecutionProvider (fp32) -- lan dau se lau, kien nhan ===")
trt = bench("TensorrtExecutionProvider", [16, 64, 256], iters=20, popts=TRT_OPTS)


## A4. Bảng so sánh Phần A


In [ ]:
hdr = "%6s | %14s %8s | %15s %8s | %6s" % ("batch", "CUDA EP pos/s", "TFLOP/s",
                                           "TensorRT pos/s", "TFLOP/s", "loi")
print(hdr)
print("-" * len(hdr))
for b in BATCHES:
    c = cuda.get(b)
    t = trt.get(b)
    cs = ("%14.1f %8.2f" % (c, c * GF)) if c else ("%14s %8s" % ("-", "-"))
    ts = ("%15.1f %8.2f" % (t, t * GF)) if t else ("%15s %8s" % ("-", "-"))
    gain = ("%5.2fx" % (t / c)) if (c and t) else "    -"
    print("%6d | %s | %s | %6s" % (b, cs, ts, gain))
print()
print("MOC SO SANH: engine dang dat ~2200 NN eval/giay = ~2.2 TFLOP/s")
print("  - Suy luan thuan tuy CAO HON NHIEU  -> engine/MCTS dang de mat hieu nang")
print("  - Xap xi 2200                       -> backend chinh la tran that su")
print("  - Cot 'loi' > 1.3x                  -> TensorRT dang co gia tri")


---
# PHẦN B — lc0 trên cùng con T4  *(tuỳ chọn)*

lc0 có **cả hai** backend: `cuda` (kernel NVIDIA viết tay) và `onnx-cuda` (chính ORT).
Chạy cả hai trên cùng máy, cùng mạng → tỉ số giữa chúng cho biết ORT đang để lại
bao nhiêu hiệu năng trên bàn.

Best-effort: nếu tải lc0 hoặc mạng thất bại thì bỏ qua, Phần A đã đủ để quyết định.


## B1. Xem lc0 có bản dựng sẵn cho Linux không


In [ ]:
import json, urllib.request
try:
    rel = json.load(urllib.request.urlopen(
        "https://api.github.com/repos/LeelaChessZero/lc0/releases/latest"))
    print("lc0 release:", rel["tag_name"])
    for a in rel["assets"]:
        print("   ", a["name"], "->", a["browser_download_url"])
except Exception as e:
    print("khong truy van duoc GitHub API:", e)


**Nếu danh sách trên có asset Linux + CUDA**, tải nó rồi chạy 4 lệnh benchmark
trong cell dưới. **Nếu không có**, bỏ qua Phần B.

Mạng cần ~1 GFLOP mỗi lượt để so được với mạng của bạn. Với bàn 8×8 của lc0:

```
FLOP ≈ 2304 × blocks × filters²
  12×192 → 1,02 GFLOP   ← khớp nhất với mạng 12×144 trên bàn 10×10
  15×192 → 1,27 GFLOP
  10×128 → 0,38 GFLOP   (quá nhỏ)
```

Lấy mạng ở <https://lczero.org/play/networks/>, chọn đúng cỡ blocks × filters.


In [ ]:
# Sua cho khop asset/mang thuc te roi bo dau # :
# !wget -q <url_asset_linux_cuda> -O lc0.tar.gz && tar xzf lc0.tar.gz
# !wget -q <url_mang> -O lc0net.pb.gz
# for be in ["cuda", "onnx-cuda", "onnx-trt", "cuda-fp16"]:
#     print("=== backend:", be, "===")
#     !./lc0 benchmark --backend={be} --weights=lc0net.pb.gz --num-positions=30
print("xem huong dan trong o markdown ben tren")


---
# PHẦN C — Cắt `ev/play` trên engine

`ev/play = 1,43` nghĩa là cứ 1 playout hữu ích thì engine gửi 1,43 thế cờ lên GPU —
**30% công việc GPU không sinh ra gì**.

Nguyên nhân là *collision*: khi gom một lô, các lần đi xuống cây sau chưa biết kết
quả của lần trước nên hay rơi trúng cùng một lá chưa được đánh giá.

Cách cắt: **gom lô nhỏ hơn** → mỗi lô ít lần đi xuống mù hơn → ít va chạm hơn.

Trước phiên đo trước, giảm lô là đánh đổi đáng sợ (batch GPU nhỏ đi). Giờ ta đã
**đo được batch to không mang lại throughput**, nên giảm lô gần như miễn phí.

5 nhánh × 4 phút ≈ 20-25 phút.


In [ ]:
import os
if not os.path.exists("/content/chess_variant_engine/custom_engine/run.sh"):
    !cd /content && rm -rf chess_variant_engine && git clone -q --depth 1 -b mcts-capacity-256 https://github.com/phuc11731510/chess_variant_engine.git
    !bash /content/chess_variant_engine/custom_engine/scripts/colab_setup.sh > /content/build.log 2>&1
    !bash /content/chess_variant_engine/custom_engine/scripts/colab_prebuilt.sh wrap
else:
    print("engine da san sang")


In [ ]:
import subprocess, re, os

ENG  = "/content/chess_variant_engine/custom_engine/run.sh"
W    = "/content/12bx144fx8s_12.onnx"
SECS = 240

# minibatch-size=0 nghia la "dung goi y cua backend" = fixed-batch.
# Lo nho hon => it lan di xuong mu hon => it collision => ev/play thap hon.
ARMS = [
    ("C1_goc_mb16", ["--fixed-batch", "16"], []),
    ("C2_mb8",      ["--fixed-batch", "8"],  ["--search-opt", "minibatch-size=8"]),
    ("C3_mb4",      ["--fixed-batch", "4"],  ["--search-opt", "minibatch-size=4"]),
    ("C4_mb2",      ["--fixed-batch", "2"],  ["--search-opt", "minibatch-size=2"]),
    ("C5_mb8_col4", ["--fixed-batch", "8"],  ["--search-opt", "minibatch-size=8",
                                              "--search-opt", "max-collision-events=4"]),
]

rows = []
for name, fb, extra in ARMS:
    out = "/content/bench_c/" + name
    subprocess.run(["rm", "-rf", out])
    os.makedirs(out, exist_ok=True)
    cmd = ["bash", ENG, "--selfplay", "--games", "100000", "--max-seconds", str(SECS),
           "--visits", "800", "--max-moves", "400", "--temp-cutoff", "32",
           "--parallel", "4", "--provider", "cuda"] + fb + extra + \
          ["--noise-alpha", "0.15", "--show-nps", "--weights", W, "--out", out]
    print(">>> %s : %s" % (name, " ".join(fb + extra)))
    r = subprocess.run(cmd, capture_output=True, text=True)
    log = r.stdout + r.stderr
    def g(pat):
        m = re.search(pat, log)
        return m.group(1) if m else "-"
    row = (name,
           g(r"Finished (\d+)/"),
           g(r"Van/gio\s*:\s*([\d.]+)"),
           g(r"NN eval/giay\s*:\s*([\d.]+)"),
           g(r"NN eval/playout\s*:\s*([\d.]+)"),
           g(r"Batch TB moi Run\(\)\s*:\s*([\d.]+)"),
           g(r"Phi do pad\s*:\s*([\d.]+)"))
    print("    van=%s  van/gio=%s  eval/s=%s  ev/play=%s  batch=%s  pad=%s%%" % row[1:])
    rows.append(row)
    subprocess.run(["rm", "-rf", out])

print()
print("%-14s%5s%10s%10s%9s%9s%8s" % ("arm", "van", "van/gio", "eval/s", "ev/play", "batchTB", "pad%"))
print("-" * 65)
for r_ in rows:
    print("%-14s%5s%10s%10s%9s%9s%8s" % r_)


---
# Gửi lại cho tôi

1. Bảng ở **A4** (CUDA EP vs TensorRT, mọi cỡ batch)
2. Bảng ở cuối **Phần C** (quét ev/play)
3. Phần B nếu chạy được

Ba thứ đó đủ để kết luận dứt khoát: engine đã kịch trần chưa, và nếu chưa thì
phần còn lại nằm ở đâu.
